In [111]:
# Rest APIs
import requests
import httpx
from urllib.request import urlopen
import time

# disable ssl verification
import ssl
ssl._create_default_https_context = ssl._create_unverified_context


### Check API Status

In [118]:
url = "http://127.0.0.1:8001"
response = requests.get(url)

print(f"Status: {response.status_code} | Elapsed: {response.elapsed.total_seconds()*1000} ms")

Status: 200 | Elapsed: 4.274 ms


### Measure Performance of http clients

In [121]:
%timeit -n 20 urlopen(url).getcode()

328 μs ± 128 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


In [102]:
%timeit -n 20 session.get(url).status_code

549 μs ± 200 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


In [120]:
%timeit -n 20 requests.get(url).status_code

603 μs ± 248 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


In [122]:
%timeit -n 20 httpx.get(url).status_code

11.8 ms ± 414 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


In [114]:
url = 'https://api.github.com'
t0 = time.time()
response = urlopen(url)
t1 = (time.time()-t0) * 1000

print(f"Status: {response.getcode()} | Elapsed: {t1} ms")

Status: 200 | Elapsed: 703.178882598877 ms


### Check Server Status

In [1]:
import socket
import time

def check_server(host, port, timeout=2):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM) #presumably
    sock.settimeout(timeout)
    t0 = time.time()
    try:
       sock.connect((host,port))
    except: 
        is_success = False
    else: 
        sock.close()
        is_success = True
    
    t1 = (time.time()-t0) * 1000
    return is_success, round(t1, 2)

In [61]:
# Ping Servers
host = "127.0.0.1"
port = 6379

status_code, elapsed = check_server(host, port)
print(f"Status: {status_code} | Elapsed: {elapsed}")


Status: True | Elapsed: 162.34


In [33]:
# SSH Servers
host = "127.0.0.1"
port = 22

status_code, elapsed = check_server(host, port)
print(f"Status: {status_code} | Elapsed: {elapsed}")

Status: True | Elapsed: 168.91


### Check Website Status

In [6]:
# Websites
host = "finanssure.com"
port = 443
status_code, elapsed = check_server(host, port)
print(f"Status: {status_code} | Elapsed: {elapsed}")

Status: True | Elapsed: 67.89


### Check Domain Expiry

In [30]:
import whois

def check_domain_expiry(domain_name):
    try:
        domain_info = whois.whois(domain_name)
        print(domain_info)
        expiry_date = domain_info.expiration_date
        print(expiry_date)

        # Handle the possibility of multiple expiration dates
        if isinstance(expiry_date, list):
            expiry_date = expiry_date[0]

        current_date = datetime.utcnow()

        if expiry_date is None:
            print(f"Could not retrieve expiration date for {domain_name}.")
        else:
            remaining_days = (expiry_date - current_date).days
            print(f"Domain {domain_name} expires on {expiry_date}.")
            print(f"Days until expiry: {remaining_days}")

            if remaining_days < 0:
                print("The domain has already expired.")
            elif remaining_days <= 30:
                print("The domain will expire soon.")
            else:
                print("The domain is valid.")

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
check_domain_expiry('finanssure.com')

{
  "domain_name": "FINANSSURE.COM",
  "registrar": "Squarespace Domains II LLC",
  "registrar_url": [
    "http://domains2.squarespace.com",
    "https://domains2.squarespace.com"
  ],
  "reseller": null,
  "whois_server": "whois.squarespace.domains",
  "referral_url": null,
  "updated_date": [
    "2024-07-01 20:54:37",
    "2024-07-01 20:54:37.131473"
  ],
  "creation_date": "2017-06-11 06:49:52",
  "expiration_date": "2025-06-11 06:49:52",
  "name_servers": [
    "DAMIAN.NS.CLOUDFLARE.COM",
    "PALOMA.NS.CLOUDFLARE.COM"
  ],
  "status": [
    "clientDeleteProhibited https://icann.org/epp#clientDeleteProhibited",
    "clientTransferProhibited https://icann.org/epp#clientTransferProhibited",
    "clientDeleteProhibited http://www.icann.org/epp#clientDeleteProhibited",
    "clientTransferProhibited http://www.icann.org/epp#clientTransferProhibited"
  ],
  "emails": "abuse-complaints@squarespace.com",
  "dnssec": "unsigned",
  "name": "REDACTED FOR PRIVACY",
  "org": null,
  "address"

### Check Certificate Expiry

In [21]:
import ssl
import socket
from datetime import datetime
import certifi

def check_certificate_expiry(hostname, port=443):
    context = ssl.create_default_context(cafile=certifi.where())

    with socket.create_connection((hostname, port)) as sock:
        with context.wrap_socket(sock, server_hostname=hostname) as ssock:
            cert = ssock.getpeercert()

    if not cert:
        print(f"Could not retrieve certificate for {hostname}")
        return

    # Get the certificate's expiration date
    exp_date_str = cert['notAfter']
    exp_date = datetime.strptime(exp_date_str, '%b %d %H:%M:%S %Y %Z')

    # Get the current date
    current_date = datetime.utcnow()

    remaining_days = (exp_date - current_date).days

    print(f"Certificate for {hostname} is valid until {exp_date}, with {remaining_days} days remaining.")

    if remaining_days < 0:
        print(f"The certificate has expired.")
    elif remaining_days <= 30:
        print(f"The certificate will expire soon: {remaining_days} days remaining.")
    else:
        print(f"The certificate is valid.")

# Example usage
hostname = 'finanssure.com'
check_certificate_expiry(hostname)

Certificate for finanssure.com is valid until 2025-03-23 20:42:23, with 51 days remaining.
The certificate is valid.


### Check Database Connection

In [22]:
# Database
from sqlalchemy import create_engine

def check_connection(conn_str):
    engine = create_engine(conn_str)
    t0 = time.time()
    try:
        conn = engine.connect()
    except: 
        is_success = False
    else: 
        conn.close()
        is_success = True
    
    t1 = (time.time()-t0) * 1000
    return is_success, round(t1, 2)

In [27]:
engine = create_engine('postgresql://POSTGRES_USER:POSTGRES_PASS@DB_HOST:DB_PORT')


In [28]:
engine.connect()

OperationalError: (psycopg2.OperationalError) could not translate host name "DB_HOST" to address: nodename nor servname provided, or not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [69]:
conn_str = 'postgresql://POSTGRES_USER:POSTGRES_PASS@DB_HOST:DB_PORT/'

status_code, elapsed = check_connection(conn_str)
print(f"Status: {status_code} | Elapsed: {elapsed}")

Status: True | Elapsed: 1911.72
